# 07 時間序列與預測：預估下週病例數與住院需求

松柏護理之家退伍軍人症群聚事件進入第二週，長官問：
> 「下禮拜還會有多少人發病？醫院還要準備幾張床？」

流程：**每日序列 → 流行曲線 + 滾動平均 → 預測 → 窗口比較 → Actual vs Predicted → 發病 vs 住院 Lag**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 建立每日發病數序列 ---
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import mean_absolute_error

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

cases = df[df["infected"] == 1]

# 每日發病數，補齊無發病的日期（確保連續）
daily = cases.groupby("symptom_onset_date").size()
daily = daily.asfreq("D", fill_value=0)
daily.name = "cases"

print(f"序列長度：{len(daily)} 天")
print(f"日期範圍：{daily.index.min().date()} \u2013 {daily.index.max().date()}")
print(f"總病例：{daily.sum()}")
print(f"\n每日病例數（前 10 天）：")
print(daily.head(10))

In [ ]:
# --- Step 2: 流行曲線 + 7 日滾動平均 ---
rolling_7 = daily.rolling(window=7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, color="#2c7fb8", alpha=0.6, label="每日新增")
ax.plot(rolling_7.index, rolling_7.values, color="red", linewidth=2,
        label="7 日滾動平均")
ax.set_title("流行曲線 + 7 日滾動平均")
ax.set_xlabel("發病日期")
ax.set_ylabel("病例數")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"\n\u2192 滾動平均可以看出趨勢：疫情何時到達高峰、何時開始下降")

In [ ]:
# --- Step 3: 滾動平均預測（3 日窗口）---
# 用前 3 天的平均值預測「下一天」的病例數
# shift(1) 非常關鍵——避免 data leakage（用到當天或未來的資料）
pred_3 = daily.rolling(window=3).mean().shift(1).dropna()
actual = daily.loc[pred_3.index]

mae_3 = mean_absolute_error(actual, pred_3)
print(f"3 日滾動平均 MAE = {mae_3:.3f}")
print(f"\u2192 平均每天的預測誤差約 {mae_3:.1f} 人")

In [ ]:
# --- Step 4: 比較不同窗口大小 ---
print("=== 不同窗口的 MAE ===")
best_w, best_mae = 3, float("inf")

for w in [3, 5, 7]:
    pred_w = daily.rolling(window=w).mean().shift(1).dropna()
    actual_w = daily.loc[pred_w.index]
    mae_w = mean_absolute_error(actual_w, pred_w)
    print(f"  window={w}  MAE={mae_w:.3f}")
    if mae_w < best_mae:
        best_w, best_mae = w, mae_w

print(f"\n\u2192 最佳窗口：window={best_w}（MAE={best_mae:.3f}）")
print("\u2192 窗口越小 \u2192 反應快但震盪大；窗口越大 \u2192 趨勢平滑但反應慢")

In [ ]:
# --- Step 5: Actual vs Predicted 對照圖 ---
pred_best = daily.rolling(window=3).mean().shift(1).dropna()
actual_best = daily.loc[pred_best.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(actual_best.index, actual_best.values, marker="o", markersize=4,
        label="實際", color="#2c7fb8")
ax.plot(pred_best.index, pred_best.values, marker="s", markersize=4,
        label="預測（3 日 MA）", color="#e34a33", linestyle="--")
ax.set_title("Actual vs Predicted（3 日滾動平均）")
ax.set_xlabel("日期")
ax.set_ylabel("每日病例數")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("\u2192 預測線追蹤實際趨勢，但在急速上升或下降時會有 lag")

In [ ]:
# --- Step 6: 發病 vs 住院曲線（Lag 效應）---
# 每日住院數
hosp_daily = (
    cases[cases["hospitalization_date"].notna()]
    .groupby("hospitalization_date").size()
)
# 對齊到相同日期範圍
all_dates = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
hosp_aligned = hosp_daily.reindex(all_dates, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, alpha=0.5, color="#2c7fb8", label="發病")
ax.bar(hosp_aligned.index, hosp_aligned.values, alpha=0.5, color="#e34a33",
       label="住院")
ax.set_title("發病 vs 住院曲線（觀察 Lag 效應）")
ax.set_xlabel("日期")
ax.set_ylabel("人數")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# 計算 lag
onset_peak = daily.idxmax()
hosp_peak = hosp_aligned.idxmax()
lag_days = (hosp_peak - onset_peak).days

print(f"發病高峰：{onset_peak.date()}")
print(f"住院高峰：{hosp_peak.date()}")
print(f"Lag = {lag_days} 天")
print(f"\n\u2192 住院高峰比發病高峰晚 {lag_days} 天")
print("\u2192 可利用此時間差提前準備床位與醫療資源")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| 每日序列 | `groupby().size()` + `asfreq('D')` 補齊日期 |
| 流行曲線 + 滾動平均 | `rolling(window=7)` 疊在 bar chart 上 |
| 滾動平均預測 | `shift(1)` 避免 data leakage |
| 窗口比較 | 不同窗口的 MAE → 選最佳 |
| Actual vs Predicted | 視覺化預測品質 |
| Lag 效應 | 發病 vs 住院的時間差 → 提前規劃床位 |

**結論**：滾動平均是最簡單的 baseline 預測模型，適合疫調現場快速估算。
住院高峰比發病高峰晚幾天，這個 lag 可幫助醫院提前準備。

下一章（Ch08），我們問「在哪裡」最嚴重？→ 空間流病。